# Pipeline NER DisTemIST con lcampillos/roberta-es-clinical-trials-ner

Entrenamiento y evaluacion de un modelo de reconocimiento de entidades clinicas (ENFERMEDAD) sobre DisTemIST.

El flujo implementa:
- Carga y preprocesamiento del dataset
- Segmentacion por oraciones y alineacion de etiquetas
- Entrenamiento k-fold multi-semilla con ensamble para inferencia
- Evaluacion en test, generacion de predicciones y evaluacion estricta por offsets

### Dependencias e importacion de librerias

Instalacion de paquetes y carga de las librerias necesarias para el pipeline.

In [1]:
%pip install -q evaluate seqeval spacy datasets transformers
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 47.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import random
import re
import time

import datasets
import evaluate
import numpy as np
import pandas as pd
import spacy
import torch

from collections import defaultdict
from pathlib import Path
from torch.optim import AdamW

from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
 )

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU disponible: True
GPU: Tesla T4


### Carga y preparacion del dataset

Se inicializan rutas, etiquetas y particiones de datos para entrenamiento y evaluacion.

In [ ]:
# Configuracion global de rutas
PROJECT_ROOT = "/kaggle/input/datasets/user"
DISTEMIST_ROOT = f"{PROJECT_ROOT}/distemist/distemist"
TEXT_FILES_DIR = f"{DISTEMIST_ROOT}/text_files"

DATA_PATHS = {
    "train_jsonl": f"{DISTEMIST_ROOT}/distemist_train.jsonl",
    "test_jsonl": f"{DISTEMIST_ROOT}/distemist_test.jsonl",
    "text_files_dir": TEXT_FILES_DIR,
    "gs_mentions_tsv": f"{DISTEMIST_ROOT}/distemist_subtrack1_test_mentions.tsv",
}

# Configuración del modelo base
BASE_MODEL = "lcampillos/roberta-es-clinical-trials-ner"

print("Rutas configuradas:")
for k, v in DATA_PATHS.items():
    print(f"  - {k}: {v}")
print(f"Modelo base: {BASE_MODEL}")

In [4]:
# Mapeo de etiquetas BIO
# Codificacion del dataset: 0=B-ENFERMEDAD, 1=I-ENFERMEDAD, 2=O
id2label = {0: "B-ENFERMEDAD", 1: "I-ENFERMEDAD", 2: "O"}
label2id = {"B-ENFERMEDAD": 0, "I-ENFERMEDAD": 1, "O": 2}
label_list = [id2label[i] for i in range(len(id2label))]

# Cargar modelo spaCy para segmentacion de oraciones
nlp_spacy = spacy.load("es_core_news_md")

# Cargar datasets JSONL
from datasets import load_dataset as _load_dataset

train_dataset = _load_dataset("json", data_files=DATA_PATHS["train_jsonl"], split="train")
test_dataset = _load_dataset("json", data_files=DATA_PATHS["test_jsonl"], split="train")

data = datasets.DatasetDict({
    "train_full": train_dataset,
    "test": test_dataset,
})

print(f"Etiquetas: {label2id}")
print(f"Train full: {len(data['train_full'])} | Test: {len(data['test'])}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Etiquetas: {'B-ENFERMEDAD': 0, 'I-ENFERMEDAD': 1, 'O': 2}
Train full: 750 | Test: 250


### Segmentacion y alineacion de etiquetas

Segmentacion por oraciones con spaCy y alineacion de etiquetas BIO durante la tokenizacion.

In [5]:
def split_by_sentences(text, tokens, labels, nlp_spacy):
    """Divide tokens y etiquetas de un documento en segmentos de oracion usando spaCy."""
    doc = nlp_spacy(text)
    sentences = list(doc.sents)

    if len(sentences) <= 1:
        return [(tokens, labels)]

    token_char_starts = []
    search_pos = 0
    for tok in tokens:
        idx = text.find(tok, search_pos)
        if idx == -1:
            return [(tokens, labels)]
        token_char_starts.append(idx)
        search_pos = idx + len(tok)

    results = []
    for sent in sentences:
        sent_start = sent.start_char
        sent_end = sent.end_char
        sent_token_indices = [
            i for i, cs in enumerate(token_char_starts)
            if sent_start <= cs < sent_end
        ]
        if not sent_token_indices:
            continue
        sent_tokens = [tokens[i] for i in sent_token_indices]
        sent_labels = [labels[i] for i in sent_token_indices]
        results.append((sent_tokens, sent_labels))

    return results if results else [(tokens, labels)]


def tokenize_and_align_labels(examples, tok, nlp_spacy, max_length=512):
    """Tokeniza por oraciones con truncation=True y propaga B->I en subtokens."""
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for doc_idx in range(len(examples["tokens"])):
        text = examples["text"][doc_idx]
        tokens = examples["tokens"][doc_idx]
        ner_tags = examples["ner_tags"][doc_idx]

        sent_chunks = split_by_sentences(text, tokens, ner_tags, nlp_spacy)

        for sent_tokens, sent_labels in sent_chunks:
            tokenized = tok(
                [sent_tokens],
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                padding=False,
            )

            word_ids = tokenized.word_ids(batch_index=0)
            previous_word_idx = None
            label_ids = []

            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(sent_labels[word_idx])
                else:
                    prev_label = sent_labels[word_idx]
                    label_ids.append(1 if prev_label == 0 else prev_label)
                previous_word_idx = word_idx

            all_input_ids.append(tokenized["input_ids"][0])
            all_attention_masks.append(tokenized["attention_mask"][0])
            all_labels.append(label_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    }

### Configuracion del experimento

Definicion de hiperparametros, modelo base y configuracion de tokenizador para entrenamiento e inferencia.

In [6]:
# --- Configuracion de experimento ---
BASE_MODEL_TAG = BASE_MODEL.split("/")[-1]

MAX_EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 8.516e-5
LR_LAYER_DECAY = 0.85
LR_ENCODER_GROUPS = 3
DROPOUT = 0.1
WEIGHT_DECAY = 0.1844
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 1e-4

K_FOLDS = 5
CV_SPLIT_SEED = 42
SEEDS = [123, 4242]
ENSEMBLE_VOTING_RATIO = 0.5

RESULTS_DIR = f"results_{BASE_MODEL_TAG}_kfold_multiseed"
MODEL_OUTPUT_PREFIX = f"{BASE_MODEL_TAG}-distemist-ner"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# --- Configuracion base de modelo/tokenizador ---
config = AutoConfig.from_pretrained(
    BASE_MODEL,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
    classifier_dropout=DROPOUT,
    attn_implementation="sdpa",
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    add_prefix_space=True,
    do_lower_case=False,
    keep_accents=True,
    
    model_max_len=config.max_position_embeddings,
)

hyperparams = {
    "base_model": BASE_MODEL,
    "base_model_tag": BASE_MODEL_TAG,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "lr_layer_decay": LR_LAYER_DECAY,
    "lr_encoder_groups": LR_ENCODER_GROUPS,
    "dropout": DROPOUT,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    "k_folds": K_FOLDS,
    "cv_split_seed": CV_SPLIT_SEED,
    "seeds": SEEDS,
    "ensemble_voting_ratio": ENSEMBLE_VOTING_RATIO,
}

with open(f"{RESULTS_DIR}/hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(hyperparams, f, ensure_ascii=False, indent=2)

print("Configuracion final cargada:")
for k, v in hyperparams.items():
    print(f"  - {k}: {v}")
print(f"Max position embeddings: {config.max_position_embeddings}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Configuracion final cargada:
  - base_model: lcampillos/roberta-es-clinical-trials-ner
  - base_model_tag: roberta-es-clinical-trials-ner
  - max_epochs: 20
  - batch_size: 16
  - learning_rate: 8.516e-05
  - lr_layer_decay: 0.85
  - lr_encoder_groups: 3
  - dropout: 0.1
  - weight_decay: 0.1844
  - warmup_ratio: 0.1
  - early_stopping_patience: 5
  - early_stopping_threshold: 0.0001
  - k_folds: 5
  - cv_split_seed: 42
  - seeds: [123, 4242]
  - ensemble_voting_ratio: 0.5
Max position embeddings: 514


### Metricas de evaluacion

Definicion de la metrica utilizada para medir el rendimiento del modelo en tareas NER.

In [7]:
metric_fn = evaluate.load("seqeval", trust_remote_code=True)


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[pred] for (pred, la) in zip(prediction, label) if la != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[la] for (_, la) in zip(prediction, label) if la != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric_fn.compute(
        predictions=true_predictions,
        references=true_labels,
        zero_division=0.0,
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### Discriminative fine-tuning

Configuracion de grupos de parametros y tasas de aprendizaje por profundidad.

In [8]:
def create_discriminative_optimizer(model):
    named_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    if not named_params:
        raise RuntimeError("No se encontraron parametros entrenables en el modelo.")

    layer_re = re.compile(r"\.(?:encoder\.layer|layer|layers|block|h)\.(\d+)\.")
    layer_ids = [int(m.group(1)) for n, _ in named_params for m in [layer_re.search(n.lower())] if m]
    max_layer_id = max(layer_ids) if layer_ids else 0

    groups = {}
    for name, param in named_params:
        lname = name.lower()

        if "classifier" in lname or "crf" in lname:
            bucket = "head"
            lr = LEARNING_RATE
        elif "embed" in lname:
            bucket = "embeddings"
            lr = LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS + 1))
        else:
            match = layer_re.search(lname)
            if match and max_layer_id > 0:
                layer_id = int(match.group(1))
                zone = min(int((layer_id / max_layer_id) * LR_ENCODER_GROUPS), LR_ENCODER_GROUPS - 1)
                bucket = f"encoder_{zone}"
                lr = LEARNING_RATE * (LR_LAYER_DECAY ** (LR_ENCODER_GROUPS - zone))
            else:
                bucket = "head"
                lr = LEARNING_RATE

        if bucket not in groups:
            groups[bucket] = {
                "params": [],
                "lr": float(lr),
                "param_count": 0,
                "tensor_count": 0,
            }
        groups[bucket]["params"].append(param)
        groups[bucket]["param_count"] += int(param.numel())
        groups[bucket]["tensor_count"] += 1

    optimizer_groups = [
        {"params": g["params"], "lr": g["lr"], "weight_decay": WEIGHT_DECAY}
        for g in groups.values()
    ]
    optimizer = AdamW(optimizer_groups, lr=LEARNING_RATE, fused=torch.cuda.is_available())

    order = ["embeddings"] + [f"encoder_{i}" for i in range(LR_ENCODER_GROUPS)] + ["head"]
    lr_group_summary = [
        {"bucket": b, "lr": groups[b]["lr"], "param_count": groups[b]["param_count"], "tensor_count": groups[b]["tensor_count"]}
        for b in order if b in groups
    ]
    return optimizer, lr_group_summary

### Entrenamiento k-fold multi-semilla

Ejecucion del entrenamiento por folds y semillas.

In [9]:
def _extract_best_eval_from_log(log_history):
    eval_logs = [
        log for log in log_history
        if "eval_f1" in log and "epoch" in log
    ]
    if not eval_logs:
        return {"best_eval_f1": np.nan, "best_epoch": np.nan}

    best_log = max(eval_logs, key=lambda x: x["eval_f1"] )
    return {
        "best_eval_f1": float(best_log["eval_f1"]),
        "best_epoch": float(best_log["epoch"]),
    }


def make_kfold_indices(n_samples, k_folds, split_seed):
    rng = np.random.default_rng(split_seed)
    all_indices = np.arange(n_samples)
    rng.shuffle(all_indices)

    fold_sizes = np.full(k_folds, n_samples // k_folds, dtype=int)
    fold_sizes[: n_samples % k_folds] += 1

    folds = []
    current = 0
    for fold_size in fold_sizes:
        val_idx = all_indices[current:current + fold_size]
        train_idx = np.concatenate((all_indices[:current], all_indices[current + fold_size:]))
        folds.append((train_idx, val_idx))
        current += fold_size

    return folds


fold_seed_results = []
ensemble_models = []
discriminative_lr_history = []

train_full_raw = data["train_full"]
folds = make_kfold_indices(len(train_full_raw), K_FOLDS, CV_SPLIT_SEED)

print("Iniciando entrenamiento k-fold multi-semilla...")
print(f"Total documentos train_full: {len(train_full_raw)}")

for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):
    train_fold_raw = train_full_raw.select(train_idx.tolist())
    val_fold_raw = train_full_raw.select(val_idx.tolist())

    train_fold_ds = train_fold_raw.map(
        lambda x: tokenize_and_align_labels(x, tokenizer, nlp_spacy, max_length=512),
        batched=True,
        remove_columns=train_fold_raw.column_names,
    )
    val_fold_ds = val_fold_raw.map(
        lambda x: tokenize_and_align_labels(x, tokenizer, nlp_spacy, max_length=512),
        batched=True,
        remove_columns=val_fold_raw.column_names,
    )

    print("\n" + "#" * 90)
    print(
        f"Fold {fold_idx}/{K_FOLDS} | "
        f"train_docs={len(train_fold_raw)} | val_docs={len(val_fold_raw)} | "
        f"train_sequences={len(train_fold_ds)} | val_sequences={len(val_fold_ds)}"
    )
    print("#" * 90)

    for seed in SEEDS:
        print("\n" + "=" * 80)
        print(f"Fold {fold_idx} | Semilla {seed} | Entrenamiento")
        print("=" * 80)

        set_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        start_time = time.time()

        model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, config=config,ignore_mismatched_sizes=True)
        model.gradient_checkpointing_enable()

        optimizer, lr_group_summary = create_discriminative_optimizer(model)

        print("Grupos LR discriminative FT:")
        for item in lr_group_summary:
            print(
                f"  - {item['bucket']}: lr={item['lr']:.8f}, "
                f"params={item['param_count']}, tensors={item['tensor_count']}"
            )

        output_dir = f"{RESULTS_DIR}/{MODEL_OUTPUT_PREFIX}-fold{fold_idx}-seed{seed}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy="epoch",
            logging_strategy="epoch",
            save_strategy="epoch",
            num_train_epochs=MAX_EPOCHS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            save_total_limit=1,
            gradient_accumulation_steps=1,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_SIZE,
            dataloader_num_workers=2,
            dataloader_prefetch_factor=4,
            dataloader_persistent_workers=True,
            seed=seed,
            bf16=True,
            save_only_model=True,
            report_to="none",
        )

        trainer_seed = Trainer(
            model,
            training_args,
            train_dataset=train_fold_ds,
            eval_dataset=val_fold_ds,
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            data_collator=DataCollatorForTokenClassification(tokenizer),
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=EARLY_STOPPING_PATIENCE,
                    early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
                )
            ],
            optimizers=(optimizer, None),
        )

        trainer_seed.train()
        model_dir = trainer_seed.state.best_model_checkpoint or output_dir

        val_metrics = trainer_seed.evaluate(val_fold_ds)
        best_info = _extract_best_eval_from_log(trainer_seed.state.log_history)
        elapsed_min = (time.time() - start_time) / 60.0

        result_row = {
            "fold": int(fold_idx),
            "seed": int(seed),
            "train_docs": int(len(train_fold_raw)),
            "val_docs": int(len(val_fold_raw)),
            "train_sequences": int(len(train_fold_ds)),
            "val_sequences": int(len(val_fold_ds)),
            "best_eval_f1": float(best_info["best_eval_f1"]),
            "best_epoch": float(best_info["best_epoch"]),
            "eval_precision": float(val_metrics.get("eval_precision", np.nan)),
            "eval_recall": float(val_metrics.get("eval_recall", np.nan)),
            "eval_f1": float(val_metrics.get("eval_f1", np.nan)),
            "eval_accuracy": float(val_metrics.get("eval_accuracy", np.nan)),
            "eval_loss": float(val_metrics.get("eval_loss", np.nan)),
            "elapsed_min": float(elapsed_min),
            "model_dir": model_dir,
        }
        fold_seed_results.append(result_row)

        discriminative_lr_history.append({
            "fold": int(fold_idx),
            "seed": int(seed),
            "model_dir": model_dir,
            "lr_groups": lr_group_summary,
        })

        ensemble_models.append({
            "fold": int(fold_idx),
            "seed": int(seed),
            "model_dir": model_dir,
            "eval_f1": float(result_row["eval_f1"]),
            "best_eval_f1": float(result_row["best_eval_f1"]),
        })

        print(
            f"Fold {fold_idx} | Semilla {seed} finalizada "
            f"| best_eval_f1={result_row['best_eval_f1']:.4f} "
            f"| eval_f1={result_row['eval_f1']:.4f} "
            f"| tiempo={elapsed_min:.1f} min"
        )

        del trainer_seed
        del model
        del optimizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if not fold_seed_results:
    raise RuntimeError("No se entreno ningun modelo fold-semilla.")

df_ensemble_results = pd.DataFrame(fold_seed_results).sort_values(
    by=["eval_f1", "best_eval_f1", "fold", "seed"],
    ascending=[False, False, True, True],
).reset_index(drop=True)

df_ensemble_results.to_csv(f"{RESULTS_DIR}/ensemble_fold_seed_summary.csv", index=False)
with open(f"{RESULTS_DIR}/ensemble_fold_seed_summary.json", "w", encoding="utf-8") as f:
    json.dump(fold_seed_results, f, ensure_ascii=False, indent=2)

with open(f"{RESULTS_DIR}/discriminative_lr_groups.json", "w", encoding="utf-8") as f:
    json.dump(discriminative_lr_history, f, ensure_ascii=False, indent=2)

ensemble_metadata = {
    "base_model": BASE_MODEL,
    "k_folds": int(K_FOLDS),
    "seeds": [int(s) for s in SEEDS],
    "ensemble_size": int(len(ensemble_models)),
    "cv_split_seed": int(CV_SPLIT_SEED),
    "lr_layer_decay": float(LR_LAYER_DECAY),
    "lr_encoder_groups": int(LR_ENCODER_GROUPS),
}
with open(f"{RESULTS_DIR}/ensemble_metadata.json", "w", encoding="utf-8") as f:
    json.dump(ensemble_metadata, f, ensure_ascii=False, indent=2)

print("\nResumen fold-semilla (top 10 por eval_f1):")
print(df_ensemble_results.head(10).to_string(index=False))
print(f"\nModelos totales en el ensamble: {len(ensemble_models)}")
print(f"Resumen LR discriminativo guardado en: {RESULTS_DIR}/discriminative_lr_groups.json")

Iniciando entrenamiento k-fold multi-semilla...
Total documentos train_full: 750


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


##########################################################################################
Fold 1/5 | train_docs=600 | val_docs=150 | train_sequences=9485 | val_sequences=2228
##########################################################################################

Fold 1 | Semilla 123 | Entrenamiento


model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Fold 5 | Semilla 4242 finalizada | best_eval_f1=0.7645 | eval_f1=0.7645 | tiempo=42.9 min

Resumen fold-semilla (top 10 por eval_f1):
 fold  seed  train_docs  val_docs  train_sequences  val_sequences  best_eval_f1  best_epoch  eval_precision  eval_recall  eval_f1  eval_accuracy  eval_loss  elapsed_min                                                                                                                          model_dir
    4   123         600       150             9400           2313      0.779693        17.0        0.778473     0.780917 0.779693       0.974160   0.371775    42.622664  results_roberta-es-clinical-trials-ner_kfold_multiseed/roberta-es-clinical-trials-ner-distemist-ner-fold4-seed123/checkpoint-3528
    4  4242         600       150             9400           2313      0.771732        15.0        0.768847     0.774639 0.771732       0.972234   0.350816    37.567711 results_roberta-es-clinical-trials-ner_kfold_multiseed/roberta-es-clinical-trials-ner-distemist-n

In [10]:
print("Resumen de validacion del ensamble:")
print(f"Modelos en ensamble: {len(ensemble_models)}")
print(f"K folds: {K_FOLDS} | Seeds: {SEEDS}")

validation_summary = {
    "eval_precision_mean": float(df_ensemble_results["eval_precision"].mean()),
    "eval_recall_mean": float(df_ensemble_results["eval_recall"].mean()),
    "eval_f1_mean": float(df_ensemble_results["eval_f1"].mean()),
    "eval_accuracy_mean": float(df_ensemble_results["eval_accuracy"].mean()),
    "eval_loss_mean": float(df_ensemble_results["eval_loss"].mean()),
    "eval_f1_std": float(df_ensemble_results["eval_f1"].std(ddof=0)),
    "ensemble_size": int(len(ensemble_models)),
}

with open(f"{RESULTS_DIR}/validation_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(validation_summary, f, ensure_ascii=False, indent=2)

for k, v in validation_summary.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"Resumen guardado en: {RESULTS_DIR}/validation_ensemble_summary.json")

Resumen de validacion del ensamble:
Modelos en ensamble: 10
K folds: 5 | Seeds: [123, 4242]
  eval_precision_mean: 0.7602
  eval_recall_mean: 0.7775
  eval_f1_mean: 0.7687
  eval_accuracy_mean: 0.9726
  eval_loss_mean: 0.3890
  eval_f1_std: 0.0057
  ensemble_size: 10
Resumen guardado en: results_roberta-es-clinical-trials-ner_kfold_multiseed/validation_ensemble_summary.json


### Artefactos de ejecucion

Consolidacion de archivos de salida y reportes generados durante el experimento.

### Resumen de validacion

Vista agregada de las metricas obtenidas en validacion para el conjunto de modelos.

In [11]:
print("Metricas agregadas de validacion (fold-semilla):")
aggregate_metrics = (
    df_ensemble_results[["eval_precision", "eval_recall", "eval_f1", "eval_accuracy", "eval_loss"]]
    .agg(["mean", "std", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

print(aggregate_metrics.to_string(index=False))
aggregate_metrics.to_csv(f"{RESULTS_DIR}/validation_ensemble_metrics_table.csv", index=False)

print(f"Tabla guardada en: {RESULTS_DIR}/validation_ensemble_metrics_table.csv")

Metricas agregadas de validacion (fold-semilla):
        metric     mean      std      min      max
eval_precision 0.760165 0.008921 0.748387 0.778473
   eval_recall 0.777515 0.007145 0.764163 0.791386
       eval_f1 0.768707 0.005970 0.756193 0.779693
 eval_accuracy 0.972586 0.001354 0.970878 0.974783
     eval_loss 0.388992 0.058220 0.276754 0.478408
Tabla guardada en: results_roberta-es-clinical-trials-ner_kfold_multiseed/validation_ensemble_metrics_table.csv


### Inferencia en test con ensamble

Aplicacion del ensamble sobre los textos de test y generacion de predicciones con offsets.

In [12]:
nlp_spacy = spacy.load("es_core_news_md")


def sentence_based_ner(texto, pipeline_ner, nlp_spacy):
    """Inferencia NER por oraciones y ajuste de offsets al documento completo."""
    doc = nlp_spacy(texto)
    all_entities = []

    for sent in doc.sents:
        sent_text = sent.text
        sent_offset = sent.start_char

        # TokenClassificationPipeline no acepta truncation/max_length en __call__
        entities = pipeline_ner(sent_text)

        for entity in entities:
            entity["start"] += sent_offset
            entity["end"] += sent_offset
            all_entities.append(entity)

    return all_entities

In [ ]:
ruta_txts = DATA_PATHS["text_files_dir"]
ruta_gs = DATA_PATHS["gs_mentions_tsv"]

print(f"Directorio de textos test: {ruta_txts}")
print(f"Gold standard: {ruta_gs}")
print(f"Modelos disponibles para ensamble: {len(ensemble_models)}")

In [14]:
texts_by_filename = {}

if not os.path.exists(ruta_txts) or len(os.listdir(ruta_txts)) == 0:
    print(f"Error: No se encuentran archivos de texto en {ruta_txts}")
else:
    for archivo in sorted(os.listdir(ruta_txts)):
        if not archivo.endswith(".txt"):
            continue
        file_path = os.path.join(ruta_txts, archivo)
        with open(file_path, "r", encoding="utf-8") as f:
            texts_by_filename[archivo.replace(".txt", "")] = f.read()

if not ensemble_models:
    raise RuntimeError("No hay modelos en el ensamble. Ejecuta primero el entrenamiento fold-semilla.")

stats = {
    "archivos_procesados": int(len(texts_by_filename)),
    "modelos_ensamblados": int(len(ensemble_models)),
    "voting_ratio": float(ENSEMBLE_VOTING_RATIO),
    "votos_requeridos": 0,
    "entidades_candidatas": 0,
    "entidades_detectadas": 0,
    "inference_total_seconds": 0.0,
    "inference_avg_model_seconds": 0.0,
    "inference_by_model": [],
}

pred_file = f"{RESULTS_DIR}/predictions_ensemble_k{K_FOLDS}_s{len(SEEDS)}.tsv"

end_to_end_start = time.time()

if not texts_by_filename:
    print("Error: No se cargaron textos de test para inferencia")
else:
    vote_threshold = max(1, int(np.ceil(ENSEMBLE_VOTING_RATIO * len(ensemble_models))))
    stats["votos_requeridos"] = int(vote_threshold)

    aggregated = defaultdict(int)
    ensemble_inference_start = time.time()

    print("Iniciando inferencia de ensamble...")
    print(f"Modelos a combinar: {len(ensemble_models)}")
    print(f"Votos requeridos por entidad: {vote_threshold}")

    for model_info in ensemble_models:
        fold = model_info["fold"]
        seed = model_info["seed"]
        model_dir = model_info["model_dir"]

        model_inference_start = time.time()
        print(f"\nInferencia con fold={fold}, seed={seed}")

        modelo_inf = AutoModelForTokenClassification.from_pretrained(model_dir)
        tokenizer_inf = AutoTokenizer.from_pretrained(model_dir)
        nlp_ner = pipeline(
            "ner",
            model=modelo_inf,
            tokenizer=tokenizer_inf,
            aggregation_strategy="simple",
        )

        for filename, texto in texts_by_filename.items():
            entidades = sentence_based_ner(texto, nlp_ner, nlp_spacy)
            for ent in entidades:
                if ent["entity_group"] != "ENFERMEDAD":
                    continue

                off0 = int(ent["start"])
                off1 = int(ent["end"])
                key = (filename, off0, off1)
                aggregated[key] += 1

        model_inference_seconds = time.time() - model_inference_start
        stats["inference_by_model"].append({
            "fold": int(fold),
            "seed": int(seed),
            "seconds": float(model_inference_seconds),
        })
        print(f"Tiempo fold={fold}, seed={seed}: {model_inference_seconds:.2f} s")

        del nlp_ner
        del tokenizer_inf
        del modelo_inf
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    stats["entidades_candidatas"] = int(len(aggregated))
    stats["inference_total_seconds"] = float(time.time() - ensemble_inference_start)
    if stats["inference_by_model"]:
        stats["inference_avg_model_seconds"] = float(
            np.mean([m["seconds"] for m in stats["inference_by_model"]])
        )

    consensus_rows = []
    for (filename, off0, off1), votes in aggregated.items():
        if votes < vote_threshold:
            continue

        texto = texts_by_filename.get(filename, "")
        consensus_rows.append({
            "filename": filename,
            "label": "ENFERMEDAD",
            "off0": off0,
            "off1": off1,
            "span": texto[off0:off1],
        })

    consensus_rows = sorted(
        consensus_rows,
        key=lambda x: (x["filename"], x["off0"], x["off1"])
)

    mark_counter = defaultdict(int)
    final_rows = []
    for row in consensus_rows:
        filename = row["filename"]
        mark_counter[filename] += 1
        final_rows.append({
            "filename": filename,
            "mark": f"T{mark_counter[filename]}",
            "label": row["label"],
            "off0": row["off0"],
            "off1": row["off1"],
            "span": row["span"],
        })

    df_pred = pd.DataFrame(
        final_rows,
        columns=["filename", "mark", "label", "off0", "off1", "span"],
    )

    if df_pred.empty:
        print("Error: DataFrame vacio tras aplicar consenso del ensamble")
    else:
        stats["entidades_detectadas"] = int(len(df_pred))
        df_pred.to_csv(pred_file, sep="\t", index=False)

        print(f"TSV generado con {len(df_pred)} entidades detectadas")
        print(f"Archivos procesados: {stats['archivos_procesados']}")
        print(f"Predicciones guardadas en: {pred_file}")
        print(f"Tiempo total inferencia ensamble: {stats['inference_total_seconds']:.2f} s")
        print(f"Tiempo medio por modelo: {stats['inference_avg_model_seconds']:.2f} s")

        print("Tiempos por modelo (fold, seed, segundos):")
        for item in stats["inference_by_model"]:
            print(f"  - fold={item['fold']}, seed={item['seed']}: {item['seconds']:.2f} s")

        with open(f"{RESULTS_DIR}/inference_stats_ensemble.json", "w", encoding="utf-8") as f:
            json.dump(stats, f, ensure_ascii=False, indent=2)

Iniciando inferencia de ensamble...
Modelos a combinar: 10
Votos requeridos por entidad: 5

Inferencia con fold=1, seed=123


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Tiempo fold=1, seed=123: 59.02 s

Inferencia con fold=1, seed=4242


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Tiempo fold=4, seed=4242: 56.24 s

Inferencia con fold=5, seed=123


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tiempo fold=5, seed=123: 55.55 s

Inferencia con fold=5, seed=4242


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tiempo fold=5, seed=4242: 55.72 s
TSV generado con 2486 entidades detectadas
Archivos procesados: 250
Predicciones guardadas en: results_roberta-es-clinical-trials-ner_kfold_multiseed/predictions_ensemble_k5_s2.tsv
Tiempo total inferencia ensamble: 565.70 s
Tiempo medio por modelo: 56.21 s
Tiempos por modelo (fold, seed, segundos):
  - fold=1, seed=123: 59.02 s
  - fold=1, seed=4242: 56.12 s
  - fold=2, seed=123: 55.69 s
  - fold=2, seed=4242: 55.54 s
  - fold=3, seed=123: 56.33 s
  - fold=3, seed=4242: 56.20 s
  - fold=4, seed=123: 55.73 s
  - fold=4, seed=4242: 56.24 s
  - fold=5, seed=123: 55.55 s
  - fold=5, seed=4242: 55.72 s


### Evaluacion estricta por offsets

Comparacion de predicciones contra la referencia mediante coincidencia exacta de etiqueta y offsets.

In [15]:
strict_eval_start = time.time()

df_gs = pd.read_csv(ruta_gs, sep="\t")
df_pred = pd.read_csv(pred_file, sep="\t")

set_gs = set(zip(df_gs["filename"], df_gs["label"], df_gs["off0"], df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp = len(set_gs.intersection(set_pred))
fp = len(set_pred - set_gs)
fn = len(set_gs - set_pred)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
fscore = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

strict_eval_seconds = time.time() - strict_eval_start
end_to_end_seconds = None
if "end_to_end_start" in globals():
    end_to_end_seconds = float(time.time() - end_to_end_start)

strict_report = {
    "base_model": BASE_MODEL,
    "k_folds": int(K_FOLDS),
    "seeds": [int(s) for s in SEEDS],
    "ensemble_size": int(len(ensemble_models)),
    "voting_ratio": float(ENSEMBLE_VOTING_RATIO),
    "tp": int(tp),
    "fp": int(fp),
    "fn": int(fn),
    "precision": float(precision),
    "recall": float(recall),
    "fscore": float(fscore),
    "strict_eval_seconds": float(strict_eval_seconds),
    "end_to_end_seconds": end_to_end_seconds,
    "predictions_file": pred_file,
}

with open(f"{RESULTS_DIR}/strict_evaluation_ensemble.json", "w", encoding="utf-8") as f:
    json.dump(strict_report, f, ensure_ascii=False, indent=2)

print(f"Modelo base:                {BASE_MODEL}")
print(f"Ensamble (folds x seeds):   {K_FOLDS} x {len(SEEDS)} = {len(ensemble_models)}")
print(f"Precision estricta:         {precision:.4f}")
print(f"Recall estricto:            {recall:.4f}")
print(f"F-score estricto:           {fscore:.4f}")
print(f"Tiempo eval. estricta:      {strict_eval_seconds:.2f} s")
if end_to_end_seconds is not None:
    print(f"Tiempo end-to-end:           {end_to_end_seconds:.2f} s")
print(f"Reporte guardado en: {RESULTS_DIR}/strict_evaluation_ensemble.json")

Modelo base:                lcampillos/roberta-es-clinical-trials-ner
Ensamble (folds x seeds):   5 x 2 = 10
Precision estricta:         0.8150
Recall estricto:            0.7798
F-score estricto:           0.7970
Tiempo eval. estricta:      0.08 s
Tiempo end-to-end:           565.92 s
Reporte guardado en: results_roberta-es-clinical-trials-ner_kfold_multiseed/strict_evaluation_ensemble.json
